In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download for your operating system.\n   - macOS: download and install the `.pkg`\n   - Windows: download and install the `.msi`\n   - Linux: run:\n   ```bash\n   curl -fsSL https://ollama.com/install.sh | sh\n   ```\n\n2. Open a terminal and start a model locally:\n```bash\nollama run llama3\n```\n\nThis downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.\n\n3. To test that the local server is running, you can use:\n```bash\ncurl http://localhost:11434\n```\n\nIf you want to use it from Python, install the client with:\n```bash\npip install ollama\n```\n\nThen call it like this:\n```python\nimport ollama\n\nresponse = ollama.chat(\n    model=\'llama3\',\n    messages=[{"role": "user", "content": your_prompt}]\n)\n\nprint(response[\'message\'][\'content\'])\n```'

In [5]:
assistant.rag("How do I run Olama locally?")

'I don’t see any FAQ entry about **running Ollama locally**.\n\nThe closest related guidance in the context is that you **can run the course locally** instead of using Codespaces if you’re comfortable setting up the needed tools such as **Python, `uv`, Jupyter, Docker, and any other module-specific tools**.\n\nIf you want, I can help you figure out the local setup based on the course tools mentioned here.'

In [6]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—probably. If the course is still open, you can usually join by enrolling through the course page or by contacting the instructor/admin if registration is closed.\n\nIf you want, I can help you figure out:\n- whether the course is still open,\n- how to enroll,\n- or draft a short message asking to join.\n\nIf you share the course name or platform, I can be more specific.'

In [4]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [5]:
search('How do I run olama?')

[{'id': 'aa310de435',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Can I run the course locally instead of Codespaces?',
  'answer': 'Yes. Codespaces is just the easiest way for everyone to start with the same environment.\n\nYou can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.\n\nIf you run locally, make sure you document your setup and keep your environment reproducible.'},
 {'id': 'e394e6f738',
  'course': 'llm-zoomcamp',
  'section': 'Workshop: Open-Source Data Ingestion (dlt)',
  'question': 'How do I know which tables are in the db?',
  'answer': 'You can use the `db.table_names()` method to list all the tables in the database.'},
 {'id': 'fe8fed31e6',
  'course': 'llm-zoomcamp',
  'section': 'Module 1 Homework',
  'question': 'How do I get token counts for Module 1 homework if I use a different provider?',
  'answer': "For the current Module 1 homework, get the token

### JSON SCHEMA

Ref: https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/13-function-calling.md

The model doesn't see our Python code, only a schema describing what the function does and what arguments it takes. LLMs are language agnostic. At the end we're just making an HTTP call, so we describe the tool in JSON rather than in Python. The same schema would work from TypeScript or Java.

In [6]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [10]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment deadline late join"}', call_id='call_Urc7LsXfcz8p3t73jlKKPLVs', name='search', type='function_call', id='fc_09598b46ca5952a4006a3b28686c7481929ecc32ee22ecab83', namespace=None, status='completed')]

In [11]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [16]:
call.name

'search'

In [17]:
args

{'query': 'join course discovered course can I join enrollment deadline late join'}

In [18]:
call.call_id

'call_Urc7LsXfcz8p3t73jlKKPLVs'

Now we send this result back to the model. First, we add the model's output to the conversation history - the model needs to see its own function call. Then we add the tool result.

In [19]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [21]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment deadline late join"}', call_id='call_Urc7LsXfcz8p3t73jlKKPLVs', name='search', type='function_call', id='fc_09598b46ca5952a4006a3b28686c7481929ecc32ee22ecab83', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_Urc7LsXfcz8p3t73jlKKPLVs',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a c

In [20]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.'

In [22]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(772, 34)

In [23]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [3]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [7]:
import json

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [ ]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course enrollment discovered course can I join"}
function_call: search {"query":"course start date enrollment open can I join late"}
function_call: search {"query":"register after course has started join course"}


In [9]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late registration FAQ"}
function_call: search {"query":"course discovered can I still join FAQ enrollment access"}
iteration #2...
ASSISTANT:
Yes — you can still join.

According to the course FAQ, you can start learning even if you just discovered the course. If you want a certificate, you need to submit your project while submissions are still open. Also, certificates are only awarded for the live cohort, not self-paced study.

If you’d like, I can also help you figure out:
- how to start the course,
- whether homework is required for the certificate,
- or what the usual workflow looks like.

Anything else you want to explore?


In [10]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results and then perform more searches.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course enroll discovered course can I join late registration FAQ"}
iteration #2...
function_call: search {"query":"certificate project while accepting submissions self-paced certificate live cohort peer-review submissions FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If your goal is just to learn, you can start anytime. If you want a certificate, you need to submit your project while the course is still accepting submissions, and certificates are only available for the live cohort, not self-paced participation.

If you’d like, I can also help you figure out how to get started with the course materials. Any other areas you want to explore?


In [11]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [12]:
agent_loop(instructions, "How do I run Olama locally?")


iteration #1...
function_call: search {"query":"Ollama local run install local model FAQ"}
iteration #2...
function_call: search {"query":"Ollama serve localhost 11434 Python client llama3 FAQ"}
iteration #3...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: install the `.pkg`
     - **Windows**: install the `.msi`
     - **Linux**: run:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This will download the model and open a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from Ollama.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )



'To run Ollama locally:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: install the `.pkg`\n     - **Windows**: install the `.msi`\n     - **Linux**: run:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and open a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from Ollama.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error later, restart the server with:\n```bash\nollama serve\n```\n\nIf you want, I can a

In [13]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discovered can I still join enroll late registration FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open.

If you’d like, I can also help you figure out the best way to catch up or start from the current week. Any other areas you want to explore?


'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.\n\nIf you’d like, I can also help you figure out the best way to catch up or start from the current week. Any other areas you want to explore?'

In [14]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening what is queen gambit"}
iteration #2...
function_call: search {"query":"queen gambit chess opening definition queen's gambit"}
iteration #3...
ASSISTANT:
You probably mean **Queen’s Gambit**.

In chess, the **Queen’s Gambit** is an opening that starts with:

1. **d4 d5**
2. **c4**

White offers the **c-pawn** to try to gain control of the center and influence the **d5** square. It’s called a “gambit” because White is offering a pawn, though Black often doesn’t keep it for long.

There are two main responses:
- **Queen’s Gambit Accepted**: Black takes the c-pawn
- **Queen’s Gambit Declined**: Black does not take it

It’s one of the most famous and respected chess openings.

If you want, I can also explain:
- why White plays c4,
- the difference between Accepted and Declined,
- or show a simple example line.


'You probably mean **Queen’s Gambit**.\n\nIn chess, the **Queen’s Gambit** is an opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** to try to gain control of the center and influence the **d5** square. It’s called a “gambit” because White is offering a pawn, though Black often doesn’t keep it for long.\n\nThere are two main responses:\n- **Queen’s Gambit Accepted**: Black takes the c-pawn\n- **Queen’s Gambit Declined**: Black does not take it\n\nIt’s one of the most famous and respected chess openings.\n\nIf you want, I can also explain:\n- why White plays c4,\n- the difference between Accepted and Declined,\n- or show a simple example line.'

### Restricting off-topic questions

In [15]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit opening chess"}
iteration #3...
ASSISTANT:
I couldn’t find any course FAQ entry about “queen gambit,” so it looks like this is off-topic for the course materials.

If you meant something course-related and can rephrase it, I can try again. Are there other areas you want to explore?


'I couldn’t find any course FAQ entry about “queen gambit,” so it looks like this is off-topic for the course materials.\n\nIf you meant something course-related and can rephrase it, I can try again. Are there other areas you want to explore?'

In [17]:
search("queen's gambit opening chess")

[{'id': '0407213397',
  'course': 'llm-zoomcamp',
  'section': 'Workshop: Open-Source Data Ingestion (dlt)',
  'question': 'There is an error when opening the table using `dbtable = db.open_table("notion_pages___homework")`: `FileNotFoundError: Table notion_pages___homework does not exist. Please first call db.create_table(notion_pages___homework, data)`',
  'answer': 'The error indicates that you have not changed all instances of "employee_handbook" to "homework" in your pipeline settings.'}]

### GUARDRIALS

To explore more

https://aishippinglabs.com/workshops/agent-with-guardrails



### FRAMEWORKS - AGENT

In [18]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [19]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [20]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [21]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [22]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [23]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [24]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


In [25]:
result.cost

CostInfo(input_cost=Decimal('0.001068'), output_cost=Decimal('0.0012915'), total_cost=Decimal('0.0023595'))

In [26]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run Ollama local setup install FAQ"}', call_id='

In [27]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [28]:
result2.cost

CostInfo(input_cost=Decimal('0.00303975'), output_cost=Decimal('0.0009135'), total_cost=Decimal('0.00395325'))

### Interactive chat

Run the built in input loop

Type questions and get answers. Type "stop" to exit.

In [29]:
runner.run()

-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"Olama run locally local install Ollama c

### Another frameworks

PydanticAI

In [45]:
from pydantic_ai import Agent

In [39]:
model = openai_client

agent_pydantic = Agent(
    model="openai:gpt-5.4-mini",
    system_prompt=instructions,
)

# Se pasa la tool

@agent_pydantic.tool_plain
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

result = agent_pydantic.run_sync("How do I run Olama locally?")
print(result.output)

To run Ollama locally, the FAQ says:

1. Install Ollama from: https://ollama.com/download  
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Start a local model by running:
   ```bash
   ollama run llama3
   ```
   This downloads the model and opens a local chat interface.

3. If you want to test that the local server is running:
   ```bash
   curl http://localhost:11434
   ```

4. If you want to use it from Python, install the client:
   ```bash
   pip install ollama
   ```

If you want, I can also help with the course-specific setup for using Ollama with the RAG homework. Are there other areas you want to explore?


In [38]:
import nest_asyncio
nest_asyncio.apply()

In [41]:
result.all_messages()

[ModelRequest(parts=[SystemPromptPart(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", timestamp=datetime.datetime(2026, 6, 24, 3, 42, 36, 484234, tzinfo=datetime.timezone.utc)), UserPromptPart(content='How do I run Olama locally?', timestamp=datetime.datetime(2026, 6, 24, 3, 42, 36, 484249, tzinfo=datetime.tim

In [44]:
result.usage

RunUsage(input_tokens=1329, output_tokens=230, details={'reasoning_tokens': 0}, requests=2, tool_calls=1)